# 📊 COVID-19 — Exploratory Data Analysis

This notebook documents the initial data exploration and validates the pipeline before wiring everything into the dashboard.
Following the scientific method: **observe → hypothesize → test → visualise**.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_and_filter
from src.preprocessing import prepare_geospatial, prepare_resource_utilization, prepare_forecasting

df = load_and_filter()
print(f'Shape: {df.shape}')
df.head()

## 1. Data Quality Audit

In [ ]:
# Null audit — important to understand before making business claims
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Null % by column:')
print(null_pct[null_pct > 0].round(1).to_string())

## 2. Geospatial Snapshot

In [ ]:
geo = prepare_geospatial(df)

fig = px.choropleth(
    geo, locations='iso_code', color='total_cases',
    hover_name='location',
    color_continuous_scale='Reds',
    title='Global Total COVID-19 Cases'
)
fig.show()

## 3. Resource Utilization — US, UK, Germany

In [ ]:
res = prepare_resource_utilization(
    df,
    countries=['United States', 'United Kingdom', 'Germany'],
    start_date='2020-03-01'
)

if 'icu_patients_per_million_7day_avg' in res.columns:
    fig = px.line(
        res, x='date', y='icu_patients_per_million_7day_avg',
        color='location',
        title='ICU Patients per Million (7-day rolling average)'
    )
    fig.show()
else:
    print('ICU column not present in this dataset slice.')

## 4. Prophet Forecast — United States

In [ ]:
from src.forecasting import run_forecast_pipeline

df_p = prepare_forecasting(df, 'United States')
print(f'Training rows: {len(df_p)}')

results = run_forecast_pipeline(df_p, forecast_horizon_days=30, run_eval=False)
forecast = results['forecast']
future   = results['future']

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_p['ds'], y=df_p['y'], name='Actual', mode='lines'))
fig.add_trace(go.Scatter(x=future['ds'], y=future['yhat'], name='Forecast',
                         line=dict(dash='dot', color='orange')))
fig.add_trace(go.Scatter(
    x=pd.concat([future['ds'], future['ds'][::-1]]),
    y=pd.concat([future['yhat_upper'], future['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(255,165,0,0.2)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI'
))
fig.update_layout(title='30-Day COVID Case Forecast — United States (Prophet)')
fig.show()

## 5. Key Findings

Document your findings here in plain English — this section is what recruiters actually read:

- **Finding 1:** ...
- **Finding 2:** ...
- **Business Recommendation:** ...